In [8]:
import pandas as pd
import unicodedata
from collections import Counter

# ============================================================
# FILES
# ============================================================

gold_file = "Are You Sure LLM Is Enough - Gold Set.csv"

prediction_files = {
    "Approach 1": "1_submission9_(approach1).csv",
    "Approach 2": "2_submission11_(approach2).csv",
    "Approach 2 + Fine-Tuned Verification": "3_submission16_(finetuned).csv",
    "ft+approach1": "4_submission16_1(fine_tune+approach1).csv",
    "Proposed Framework": "5_submission_16(2)_(finetuned+approach1+postprocess).csv"
}


# ============================================================
# TOKEN NORMALIZATION
# ============================================================

def normalize_text(text):
    """
    Normalize text before token-level F1 calculation.
    Punctuation is REPLACED WITH A SPACE (not deleted) so that
    hyphenated / comma-joined tokens don't get glued together
    into a single non-matching token. Whitespace is then collapsed.
    """
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    text = "".join(
        ch if not unicodedata.category(ch).startswith("P") else " "
        for ch in text
    )

    return " ".join(text.split())


# ============================================================
# TOKEN-LEVEL F1
# ============================================================

def token_f1(prediction, ground_truth):

    prediction = normalize_text(prediction)
    ground_truth = normalize_text(ground_truth)

    pred_tokens = prediction.split()
    gold_tokens = ground_truth.split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    pred_counter = Counter(pred_tokens)
    gold_counter = Counter(gold_tokens)

    overlap = sum((pred_counter & gold_counter).values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)

    f1 = 2 * precision * recall / (precision + recall)

    return f1


# ============================================================
# LOAD GOLD SET
# ============================================================

gold = pd.read_csv(gold_file)

print("Gold Set:")
print("Rows:", len(gold))
print("Columns:", gold.columns.tolist())

print("\nUsage distribution:")
print(gold["Usage"].value_counts())


# ============================================================
# CALCULATE RESULTS
# ============================================================

results = []

for method, filename in prediction_files.items():

    print("\n" + "=" * 60)
    print(method)
    print("=" * 60)

    pred = pd.read_csv(filename)

    print("Prediction rows:", len(pred))

    merged = gold[["index", "answer", "Usage"]].merge(
        pred[["index", "answer"]],
        on="index",
        how="inner",
        suffixes=("_gold", "_pred")
    )

    if len(merged) != len(gold):
        raise ValueError(
            f"{method}: Only {len(merged)} of {len(gold)} "
            "questions matched."
        )

    merged["F1"] = merged.apply(
        lambda row: token_f1(
            row["answer_pred"],
            row["answer_gold"]
        ),
        axis=1
    )

    overall_f1 = merged["F1"].mean()
    public_f1 = merged.loc[merged["Usage"] == "Public", "F1"].mean()
    private_f1 = merged.loc[merged["Usage"] == "Private", "F1"].mean()

    results.append({
        "Method": method,
        "Overall F1": overall_f1,
        "Public F1": public_f1,
        "Private F1": private_f1
    })


# ============================================================
# FINAL TABLE
# ============================================================

results_df = pd.DataFrame(results)

print("\n\n")
print("=" * 80)
print("FINAL TOKEN-LEVEL F1 RESULTS")
print("=" * 80)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

Gold Set:
Rows: 1500
Columns: ['index', 'question', 'answer', 'Usage']

Usage distribution:
Usage
Public     1032
Private     468
Name: count, dtype: int64

Approach 1
Prediction rows: 1500

Approach 2
Prediction rows: 1500

Approach 2 + Fine-Tuned Verification
Prediction rows: 1500

ft+approach1
Prediction rows: 1500

Proposed Framework
Prediction rows: 1500



FINAL TOKEN-LEVEL F1 RESULTS
                              Method  Overall F1  Public F1  Private F1
                          Approach 1     0.69268    0.69327     0.69136
                          Approach 2     0.70124    0.70224     0.69903
Approach 2 + Fine-Tuned Verification     0.71865    0.71585     0.72484
                        ft+approach1     0.72021    0.71649     0.72840
                  Proposed Framework     0.72040    0.71649     0.72901


In [3]:
import pandas as pd
import unicodedata
from collections import Counter

# ============================================================
# FILES
# ============================================================

gold_file = "Are You Sure LLM Is Enough - Gold Set.csv"

# Edit these paths/names to match your actual CSV files
prediction_files = {
    "Gemma-4-E2B-it": "e2b_pred.csv",
    "Gemma-4-E4B-it": "e4b_pred.csv",
    "Gemma-4-26B-A4B-it": "e26b_pred.csv",
}

# ============================================================
# TOKEN NORMALIZATION (same as main eval script)
# ============================================================

def normalize_text(text):
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = "".join(
        ch if not unicodedata.category(ch).startswith("P") else " "
        for ch in text
    )
    return " ".join(text.split())


def token_f1(prediction, ground_truth):
    prediction = normalize_text(prediction)
    ground_truth = normalize_text(ground_truth)

    pred_tokens = prediction.split()
    gold_tokens = ground_truth.split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    pred_counter = Counter(pred_tokens)
    gold_counter = Counter(gold_tokens)
    overlap = sum((pred_counter & gold_counter).values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


# ============================================================
# LOAD GOLD SET
# ============================================================

gold = pd.read_csv(gold_file)

print("Gold Set:")
print("Rows:", len(gold))
print("Usage distribution:")
print(gold["Usage"].value_counts())

# ============================================================
# CALCULATE RESULTS FOR EACH MODEL
# ============================================================

results = []

for method, filename in prediction_files.items():

    print("\n" + "=" * 60)
    print(method)
    print("=" * 60)

    pred = pd.read_csv(filename)
    print("Prediction rows:", len(pred))

    merged = gold[["index", "answer", "Usage"]].merge(
        pred[["index", "answer"]],
        on="index",
        how="inner",
        suffixes=("_gold", "_pred")
    )

    if len(merged) != len(gold):
        raise ValueError(
            f"{method}: Only {len(merged)} of {len(gold)} "
            "questions matched."
        )

    merged["F1"] = merged.apply(
        lambda row: token_f1(row["answer_pred"], row["answer_gold"]),
        axis=1
    )

    overall_f1 = merged["F1"].mean()
    public_f1 = merged.loc[merged["Usage"] == "Public", "F1"].mean()
    private_f1 = merged.loc[merged["Usage"] == "Private", "F1"].mean()

    results.append({
        "Model": method,
        "Overall F1": overall_f1,
        "Public F1": public_f1,
        "Private F1": private_f1
    })

# ============================================================
# FINAL TABLE
# ============================================================

results_df = pd.DataFrame(results)

print("\n\n")
print("=" * 80)
print("MODEL SCALING RESULTS")
print("=" * 80)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

results_df.to_csv("model_scaling_results.csv", index=False)
print("\nSaved: model_scaling_results.csv")

Gold Set:
Rows: 1500
Usage distribution:
Usage
Public     1032
Private     468
Name: count, dtype: int64

Gemma-4-E2B-it
Prediction rows: 1500

Gemma-4-E4B-it
Prediction rows: 1500

Gemma-4-26B-A4B-it
Prediction rows: 1500



MODEL SCALING RESULTS
             Model  Overall F1  Public F1  Private F1
    Gemma-4-E2B-it     0.60031    0.59636     0.60901
    Gemma-4-E4B-it     0.60395    0.60244     0.60728
Gemma-4-26B-A4B-it     0.66911    0.66388     0.68065

Saved: model_scaling_results.csv


In [9]:
import pandas as pd
import re

# ============================================================
# FILES
# ============================================================

prediction_files = {
    "Approach 1": "1_submission9_(approach1).csv",
    "Approach 2": "2_submission11_(approach2).csv",
    "Approach 2 + Fine-Tuned Verification": "3_submission16_(finetuned).csv",
    "ft+approach1": "4_submission16_1(fine_tune+approach1).csv",
    "Proposed Framework": "5_submission_16(2)_(finetuned+approach1+postprocess).csv"
}



# ============================================================
# ABSTENTION / UNRESOLVED ANSWER PATTERNS
# ============================================================

abstention_pattern = re.compile(
    r"""
    # Direct abstention
    তথ্য\s*নেই
    |
    তথ্যে\s*নেই
    |
    উল্লেখ\s*নেই
    |
    
    # Context-based abstention
    context\s*[-–—]?\s*এ\s*তথ্য\s*নেই
    |
    context\s*[-–—]?\s*এ\s*উল্লেখ\s*নেই
    |
    
    # Romanized form
    context\s+e\s+tottho\s+nei
    |
    tottho\s+nei
    |
    
    # Longer abstention responses
    প্রদত্ত\s+তথ্যে.*নেই
    |
    প্রদত্ত\s+তথ্যে.*উল্লেখ\s*নেই
    |
    প্রদত্ত\s+context.*নেই
    |
    প্রদত্ত\s+context.*উল্লেখ\s*নেই
    """,
    re.IGNORECASE | re.VERBOSE
)


# ============================================================
# FIND UNRESOLVED ANSWERS
# ============================================================

all_results = []

for method, filename in prediction_files.items():

    df = pd.read_csv(filename)

    # Convert answer to string
    answers = df["answer"].fillna("").astype(str)

    # Detect abstention-style answers
    mask = answers.apply(
        lambda x: bool(abstention_pattern.search(x))
    )

    unresolved = df[mask].copy()

    all_results.append({
        "Method": method,
        "Unresolved": len(unresolved)
    })

    print("\n" + "=" * 70)
    print(method)
    print("=" * 70)

    print("Total predictions:", len(df))
    print("Unresolved predictions:", len(unresolved))

    print("\nPatterns found:")
    print(unresolved["answer"].value_counts())

    print("\nUnresolved indices:")
    print(unresolved["index"].tolist())


# ============================================================
# SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame(all_results)

print("\n\n" + "=" * 70)
print("UNRESOLVED PREDICTIONS SUMMARY")
print("=" * 70)

print(summary_df.to_string(index=False))


Approach 1
Total predictions: 1500
Unresolved predictions: 24

Patterns found:
answer
Context-এ তথ্য নেই    24
Name: count, dtype: int64

Unresolved indices:
['test_0042', 'test_0100', 'test_0200', 'test_0211', 'test_0337', 'test_0350', 'test_0361', 'test_0435', 'test_0463', 'test_0481', 'test_0497', 'test_0503', 'test_0520', 'test_0637', 'test_0643', 'test_0692', 'test_0697', 'test_0823', 'test_0833', 'test_0894', 'test_0957', 'test_1104', 'test_1161', 'test_1414']

Approach 2
Total predictions: 1500
Unresolved predictions: 12

Patterns found:
answer
তথ্য নেই                                                                                                                                                                        4
Context-এ উল্লেখ নেই                                                                                                                                                            1
উল্লেখ নেই                                                                           

In [2]:
import pandas as pd
import unicodedata
from collections import Counter

# ============================================================
# FILES
# ============================================================

gold_file = "Are You Sure LLM Is Enough - Gold Set.csv"
prediction_file = "5_submission_16(2)_(finetuned+approach1+postprocess).csv"  # Proposed Framework

# ============================================================
# TOKEN NORMALIZATION (unchanged from your eval script)
# ============================================================

def normalize_text(text):
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = "".join(
        ch if not unicodedata.category(ch).startswith("P") else " "
        for ch in text
    )
    return " ".join(text.split())


def token_f1_detailed(prediction, ground_truth):
    """
    Same as token_f1, but returns precision, recall, and F1
    so we can report the breakdown per example.
    """
    pred_norm = normalize_text(prediction)
    gold_norm = normalize_text(ground_truth)

    pred_tokens = pred_norm.split()
    gold_tokens = gold_norm.split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0, 1.0, 1.0

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0, 0.0, 0.0

    pred_counter = Counter(pred_tokens)
    gold_counter = Counter(gold_tokens)
    overlap = sum((pred_counter & gold_counter).values())

    if overlap == 0:
        return 0.0, 0.0, 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


# ============================================================
# LOAD DATA
# ============================================================

gold = pd.read_csv(gold_file)
pred = pd.read_csv(prediction_file)

merged = gold[["index", "answer"]].merge(
    pred[["index", "answer"]],
    on="index",
    how="inner",
    suffixes=("_gold", "_pred")
)

# ============================================================
# COMPUTE PRECISION / RECALL / F1 PER QUESTION
# ============================================================

precisions, recalls, f1s = [], [], []
for _, row in merged.iterrows():
    p, r, f = token_f1_detailed(row["answer_pred"], row["answer_gold"])
    precisions.append(p)
    recalls.append(r)
    f1s.append(f)

merged["Precision"] = precisions
merged["Recall"] = recalls
merged["F1"] = f1s

# ============================================================
# CATEGORIZE INTO QUALITATIVE BUCKETS
# ============================================================

def categorize(f1):
    if f1 == 1.0:
        return "Perfect Match"
    elif f1 >= 0.7:
        return "High Overlap"
    elif f1 > 0.0:
        return "Partial Match"
    else:
        return "No Match"

merged["Category"] = merged["F1"].apply(categorize)

print("Category distribution:")
print(merged["Category"].value_counts())
print()

# ============================================================
# PICK ONE REPRESENTATIVE EXAMPLE PER CATEGORY
# ============================================================

sample_rows = []
for cat in ["Perfect Match", "High Overlap", "Partial Match", "No Match"]:
    subset = merged[merged["Category"] == cat]
    if len(subset) == 0:
        continue
    # pick a mid-length example (avoids picking a trivially short/degenerate one)
    subset = subset.copy()
    subset["gold_len"] = subset["answer_gold"].astype(str).str.len()
    subset = subset.sort_values("gold_len")
    example = subset.iloc[len(subset) // 2]
    sample_rows.append(example)

examples_df = pd.DataFrame(sample_rows)

# ============================================================
# PRINT TABLE FOR PAPER (LaTeX-ready row format)
# ============================================================

print("=" * 90)
print("QUALITATIVE F1 EXAMPLES (for paper Table)")
print("=" * 90)

for _, row in examples_df.iterrows():
    print(f"\nCategory: {row['Category']}")
    print(f"Ground Truth: {row['answer_gold']}")
    print(f"Prediction:   {row['answer_pred']}")
    print(f"Precision={row['Precision']:.3f}  Recall={row['Recall']:.3f}  F1={row['F1']:.3f}")

# Optional: dump as CSV so you can copy Bangla text into the LaTeX table directly
examples_df[["Category", "answer_gold", "answer_pred", "Precision", "Recall", "F1"]].to_csv(
    "qualitative_f1_examples.csv", index=False
)
print("\nSaved: qualitative_f1_examples.csv")

Category distribution:
Category
Perfect Match    793
Partial Match    364
No Match         218
High Overlap     125
Name: count, dtype: int64

QUALITATIVE F1 EXAMPLES (for paper Table)

Category: Perfect Match
Ground Truth: পার্ক সুং-জিন।
Prediction:   পার্ক সুং-জিন
Precision=1.000  Recall=1.000  F1=1.000

Category: High Overlap
Ground Truth: যোগের লক্ষ্যগুলি তৈরি করে না।
Prediction:   ভঙ্গিতে দক্ষতা যোগের লক্ষ্যগুলি তৈরি করে না
Precision=0.714  Recall=1.000  F1=0.833

Category: Partial Match
Ground Truth: প্রাক্তন বোলার।
Prediction:   ভারতের জাতীয় দলের প্রাক্তন বোলার
Precision=0.400  Recall=1.000  F1=0.571

Category: No Match
Ground Truth: সম্পাদন করে।
Prediction:   ১ বাইট দৈর্ঘ্যের অপকোড এবং প্রয়োজনীয় প্যারামিটার ব্যবহারের মাধ্যমে
Precision=0.000  Recall=0.000  F1=0.000

Saved: qualitative_f1_examples.csv
